# SciCite Citation-Cue ToW API Annotation (CC-ToW)

This notebook generates the cached **Citation-Cue ToW (CC-ToW)** annotation files used in the SciCite extension experiments.

CC-ToW is the task-oriented extension variant: for each citation context, the annotation model identifies cue words or short phrases and writes a concise explanation of how those cues indicate the citation's function.

This notebook performs **data generation only**. It does not train or evaluate classifiers. The clean **Inline Word-Level ToW (IW-ToW)** dataset is generated separately in `03_scicite_inline_wordlevel_tow_generator.ipynb`, because the final paper uses the stricter target-word filtering from that notebook.


## 1. Install dependencies


In [ ]:
!pip install -q "openai>=1.0.0" "pandas==2.2.2" "numpy==2.0.2"


## 2. Mount Drive and configure paths

The output paths are intentionally stable so downstream notebooks can load the generated annotation files without manual edits.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from collections import Counter
from getpass import getpass
import os, json, random, time, re, tarfile, urllib.request, shutil
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/tow_project/B/extension/scicite")
DATA_DIR = PROJECT_ROOT / "data"
CC_TOW_DIR = PROJECT_ROOT / "tow_generations"
DOWNLOAD_DIR = PROJECT_ROOT / "official_scicite_download"

for d in [PROJECT_ROOT, DATA_DIR, CC_TOW_DIR, DOWNLOAD_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("CC_TOW_DIR:", CC_TOW_DIR)


## 3. Generation configuration

Set `GENERATION_LIMITS` to small values for a smoke test. Set each split to `None` for full generation over the sampled SciCite splits.


In [ ]:
# Raw SciCite sample sizes used by the extension experiments.
TRAIN_N = 300
VAL_N = 100
TEST_N = 100
FORCE_RESAMPLE = False

SEED = 42
random.seed(SEED)

# API model used for annotation. Change here if needed.
OPENAI_MODEL = "gpt-5.4-mini"
MAX_RETRIES = 4
SLEEP_BASE_SECONDS = 2
API_THROTTLE_SECONDS = 0.0

GENERATE_SPLITS = ["train", "validation", "test"]
# Example smoke test: {"train": 5, "validation": 5, "test": 5}
GENERATION_LIMITS = {"train": None, "validation": None, "test": None}

RUN_CC_TOW_GENERATION = True

# Stable label order for SciCite.
LABEL_NAMES = ["background", "method", "result"]
LABEL_TO_ID = {name: i for i, name in enumerate(LABEL_NAMES)}

print("OPENAI_MODEL:", OPENAI_MODEL)
print("GENERATION_LIMITS:", GENERATION_LIMITS)


## 4. JSONL helpers


In [ ]:
def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def append_jsonl(row, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

def write_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def write_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


## 5. Load and sample SciCite

SciCite is loaded from the official AllenAI JSONL tarball rather than `load_dataset("allenai/scicite")`, because newer versions of Hugging Face `datasets` no longer support some older dataset scripts.


In [ ]:
# ============================================================
# Load SciCite from the official AllenAI tarball and save sampled raw splits
# ============================================================
# Why not load_dataset("allenai/scicite")?
# Newer Hugging Face datasets versions reject older dataset scripts like scicite.py.
# To avoid that failure, this notebook downloads the official SciCite JSONL tarball
# linked from https://github.com/allenai/scicite and reads the JSONL files directly.

import tarfile
import urllib.request
from pathlib import Path

SCICITE_URL = "https://s3-us-west-2.amazonaws.com/ai2-s2-research/scicite/scicite.tar.gz"
SCICITE_CACHE_DIR = EXT_DIR / "official_scicite_download"
SCICITE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
TARBALL_PATH = SCICITE_CACHE_DIR / "scicite.tar.gz"
EXTRACT_DIR = SCICITE_CACHE_DIR / "extracted"

if not TARBALL_PATH.exists():
    print("Downloading official SciCite tarball...")
    print(SCICITE_URL)
    urllib.request.urlretrieve(SCICITE_URL, TARBALL_PATH)
else:
    print("Reusing downloaded tarball:", TARBALL_PATH)

if not EXTRACT_DIR.exists() or not list(EXTRACT_DIR.rglob("*.jsonl")):
    print("Extracting SciCite tarball...")
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with tarfile.open(TARBALL_PATH, "r:gz") as tar:
        tar.extractall(EXTRACT_DIR)
else:
    print("Reusing extracted SciCite files:", EXTRACT_DIR)

jsonl_files = sorted(EXTRACT_DIR.rglob("*.jsonl"))
print("Found JSONL files:")
for p in jsonl_files:
    print(" -", p.relative_to(EXTRACT_DIR))

if not jsonl_files:
    raise RuntimeError("No .jsonl files found after extracting SciCite tarball.")

def choose_split_file(split):
    candidates = []
    for p in jsonl_files:
        name = p.name.lower()
        full = str(p).lower()
        if split == "train" and "train" in name:
            candidates.append(p)
        elif split == "validation" and any(x in name for x in ["dev", "valid", "validation"]):
            candidates.append(p)
        elif split == "test" and "test" in name:
            candidates.append(p)
    if not candidates:
        raise RuntimeError(f"Could not find a {split} JSONL file. Available: {[str(p.relative_to(EXTRACT_DIR)) for p in jsonl_files]}")
    # Prefer files directly named train/dev/test over auxiliary files.
    candidates = sorted(candidates, key=lambda p: (len(p.name), str(p)))
    return candidates[0]

SPLIT_FILES = {
    "train": choose_split_file("train"),
    "validation": choose_split_file("validation"),
    "test": choose_split_file("test"),
}
print("\nUsing split files:")
for split, p in SPLIT_FILES.items():
    print(split, "->", p.relative_to(EXTRACT_DIR))

# Official SciCite labels are string labels. Keep this stable order throughout the notebook.
LABEL_NAMES = ["background", "method", "result"]
LABEL_TO_ID = {name: i for i, name in enumerate(LABEL_NAMES)}
print("Label names:", LABEL_NAMES)

def read_official_scicite_file(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

raw_scicite = {split: read_official_scicite_file(path) for split, path in SPLIT_FILES.items()}
print("\nOfficial split sizes:", {k: len(v) for k, v in raw_scicite.items()})

def get_text_field(example):
    # Official SciCite uses "string". Some exports also expose "context".
    for key in ["string", "context", "text", "citation", "sentence"]:
        if key in example and example[key] is not None and str(example[key]).strip():
            return str(example[key]).strip()
    raise KeyError(f"No text field found. Available keys: {list(example.keys())}")

def normalize_label(label):
    if isinstance(label, int):
        if 0 <= label < len(LABEL_NAMES):
            return label, LABEL_NAMES[label]
        raise ValueError(f"Unknown numeric label: {label}")
    label_name = str(label).strip().lower().replace("_", " ")
    # Common normalization just in case.
    if label_name in ["result comparison", "result_comparison", "results"]:
        label_name = "result"
    if label_name not in LABEL_TO_ID:
        raise ValueError(f"Unknown label {label!r}. Expected one of {LABEL_NAMES}")
    return LABEL_TO_ID[label_name], label_name

def stratified_sample(rows, n, seed=42):
    rng = random.Random(seed)
    label_to_indices = {}
    for i, ex in enumerate(rows):
        label_id, _ = normalize_label(ex["label"])
        label_to_indices.setdefault(label_id, []).append(i)

    labels = sorted(label_to_indices.keys())
    per_label = n // len(labels)
    remainder = n % len(labels)

    selected = []
    for j, label in enumerate(labels):
        indices = label_to_indices[label][:]
        rng.shuffle(indices)
        take = per_label + (1 if j < remainder else 0)
        selected.extend(indices[:min(take, len(indices))])

    if len(selected) < n:
        already = set(selected)
        remaining = [i for i in range(len(rows)) if i not in already]
        rng.shuffle(remaining)
        selected.extend(remaining[:n - len(selected)])

    rng.shuffle(selected)
    return [rows[i] for i in selected[:n]]

def convert_record(example, idx, split):
    label_id, label_name = normalize_label(example["label"])
    return {
        "id": f"{split}_{idx}",
        "source_split": split,
        "text": get_text_field(example),
        "label": label_id,
        "label_name": label_name,
        # Keep a little original metadata for traceability, if present.
        "citingPaperId": example.get("citingPaperId"),
        "citedPaperId": example.get("citedPaperId"),
        "sectionName": example.get("sectionName"),
    }

def save_jsonl(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    with open(tmp_path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    tmp_path.replace(path)
    print(f"Wrote {len(records)} records -> {path}")

def maybe_write_sample(split, sampled_rows):
    out_path = DATA_DIR / f"scicite_{split}_raw_sample.jsonl"
    if out_path.exists() and not FORCE_RESAMPLE:
        print(f"Reusing existing sample: {out_path}")
        return
    records = [convert_record(ex, i, split) for i, ex in enumerate(sampled_rows)]
    save_jsonl(records, out_path)

samples = {
    "train": stratified_sample(raw_scicite["train"], TRAIN_N, SEED),
    "validation": stratified_sample(raw_scicite["validation"], VAL_N, SEED),
    "test": stratified_sample(raw_scicite["test"], TEST_N, SEED),
}

for split, sampled_rows in samples.items():
    maybe_write_sample(split, sampled_rows)

# Save metadata needed after runtime restarts.
with open(DATA_DIR / "label_names.json", "w", encoding="utf-8") as f:
    json.dump(LABEL_NAMES, f, indent=2)

manifest = {
    "source": "official_allenai_scicite_tarball",
    "source_url": SCICITE_URL,
    "seed": SEED,
    "train_n": TRAIN_N,
    "validation_n": VAL_N,
    "test_n": TEST_N,
    "force_resample": FORCE_RESAMPLE,
    "label_names": LABEL_NAMES,
    "split_files": {k: str(v.relative_to(EXTRACT_DIR)) for k, v in SPLIT_FILES.items()},
}
with open(DATA_DIR / "sampling_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\nSaved files:")
for p in sorted(DATA_DIR.glob("*.jsonl")):
    print(p.name, "-", round(p.stat().st_size / 1024, 2), "KB")

print("\nPreview training example:")
with open(DATA_DIR / "scicite_train_raw_sample.jsonl", "r", encoding="utf-8") as f:
    print(json.dumps(json.loads(next(f)), indent=2, ensure_ascii=False))


## 6. OpenAI API setup

The notebook first checks Colab Secrets for `OPENAI_API_KEY`. If the key is not present, it prompts for hidden manual input.


In [ ]:
from openai import OpenAI

try:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY")
    if key:
        os.environ["OPENAI_API_KEY"] = key
except Exception:
    pass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")

client = OpenAI()
print("OpenAI client ready.")


## 7. Generate Citation-Cue ToW (CC-ToW)

CC-ToW is a task-oriented annotation variant. It identifies cue words or short phrases in the citation context and explains how those cues indicate the citation's rhetorical role, without revealing or naming the gold label.


In [ ]:
CC_TOW_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "cue_words": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 3,
            "maxItems": 8,
        },
        "tow_explanation": {
            "type": "string",
            "description": "One or two sentences explaining how cue words indicate the citation's role.",
        },
    },
    "required": ["cue_words", "tow_explanation"],
}

CC_SYSTEM_PROMPT = """
You generate brief Citation-Cue Thoughts of Words (CC-ToW) annotations for scientific citation contexts.

Goal:
- Identify important cue words or short phrases in the citation context.
- Explain how those visible words suggest the function of the citation.
- Do not predict, reveal, or name a SciCite label.
- Do not use the gold label.
- Keep the explanation brief and grounded in the visible citation context.
- Return JSON only.
""".strip()

def extract_json_object(text):
    text = (text or "").strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in response: {text[:500]}")
    return json.loads(match.group(0))

def responses_json_call(model, system_prompt, user_prompt, schema, schema_name, max_output_tokens=300):
    response = client.responses.create(
        model=model,
        input=[
            {"role": "developer", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": schema_name,
                "schema": schema,
                "strict": True,
            }
        },
        max_output_tokens=max_output_tokens,
        store=False,
    )
    return json.loads(response.output_text)

def generate_cc_tow_for_text(text, model=OPENAI_MODEL):
    user_prompt = f"""
Citation context:
{text}

Return a JSON object with:
- cue_words: 3 to 8 important words or short phrases copied or closely paraphrased from the citation.
- tow_explanation: 1 to 2 sentences explaining how these cues connect the citation context to the citation's role/function, without naming a SciCite label.
""".strip()

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return responses_json_call(
                model=model,
                system_prompt=CC_SYSTEM_PROMPT,
                user_prompt=user_prompt,
                schema=CC_TOW_SCHEMA,
                schema_name="cc_tow_annotation",
                max_output_tokens=300,
            )
        except Exception as e:
            last_err = e
            try:
                response = client.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": CC_SYSTEM_PROMPT},
                        {"role": "user", "content": user_prompt},
                    ],
                    response_format={"type": "json_object"},
                    max_tokens=300,
                )
                return extract_json_object(response.choices[0].message.content)
            except Exception as e2:
                last_err = e2
                print(f"CC-ToW attempt {attempt}/{MAX_RETRIES} failed: {repr(last_err)}")
                time.sleep(SLEEP_BASE_SECONDS * attempt)
    raise RuntimeError(f"CC-ToW generation failed after {MAX_RETRIES} attempts. Last error: {repr(last_err)}")

def validate_cc_tow_row(row):
    return (
        isinstance(row, dict)
        and isinstance(row.get("id"), str)
        and isinstance(row.get("text"), str)
        and row.get("label") is not None
        and isinstance(row.get("cue_words"), list)
        and len(row.get("cue_words", [])) >= 3
        and isinstance(row.get("tow"), str)
        and len(row.get("tow", "").strip()) > 20
    )

def generate_cc_tow_split(split, limit=None):
    raw_path = DATA_DIR / f"scicite_{split}_raw_sample.jsonl"
    out_path = CC_TOW_DIR / f"scicite_{split}_tow.jsonl"
    progress_path = CC_TOW_DIR / f"scicite_{split}_tow_progress.json"

    raw_rows = read_jsonl(raw_path)
    existing_rows = read_jsonl(out_path)
    existing_valid_ids = {r.get("id") for r in existing_rows if validate_cc_tow_row(r)}

    print(f"\n=== CC-ToW {split} ===")
    print("raw rows:", len(raw_rows), "existing valid:", len(existing_valid_ids))

    generated = 0
    skipped = 0
    errors = 0
    for row in raw_rows:
        if limit is not None and generated >= limit:
            break
        if row["id"] in existing_valid_ids:
            skipped += 1
            continue
        try:
            ann = generate_cc_tow_for_text(row["text"])
            out = dict(row)
            out.update({
                "cue_words": ann.get("cue_words", []),
                "tow": ann.get("tow_explanation", "").strip(),
                "annotation_type": "CC-ToW",
                "openai_model": OPENAI_MODEL,
            })
            if not validate_cc_tow_row(out):
                raise ValueError(f"Invalid CC-ToW row: {out}")
            append_jsonl(out, out_path)
            existing_valid_ids.add(row["id"])
            generated += 1
            if API_THROTTLE_SECONDS:
                time.sleep(API_THROTTLE_SECONDS)
        except Exception as e:
            errors += 1
            append_jsonl({"id": row.get("id"), "split": split, "error": repr(e)}, CC_TOW_DIR / f"{split}_cc_tow_errors.jsonl")
            print("Error:", row.get("id"), repr(e))

    final_valid = [r for r in read_jsonl(out_path) if validate_cc_tow_row(r)]
    write_json({
        "split": split,
        "raw_total": len(raw_rows),
        "valid_generated": len(final_valid),
        "generated_this_run": generated,
        "skipped_existing": skipped,
        "errors_this_run": errors,
        "model": OPENAI_MODEL,
        "path": str(out_path),
    }, progress_path)
    print("valid CC-ToW rows:", len(final_valid), "->", out_path)
    return out_path

if RUN_CC_TOW_GENERATION:
    for split in GENERATE_SPLITS:
        generate_cc_tow_split(split, GENERATION_LIMITS.get(split))
else:
    print("RUN_CC_TOW_GENERATION=False; skipping CC-ToW generation.")


## 8. Summarize CC-ToW outputs


In [ ]:
cc_summary = []
for split in GENERATE_SPLITS:
    path = CC_TOW_DIR / f"scicite_{split}_tow.jsonl"
    rows = read_jsonl(path)
    valid = [r for r in rows if validate_cc_tow_row(r)]
    cc_summary.append({
        "split": split,
        "rows": len(rows),
        "valid_cc_tow": len(valid),
        "labels": dict(Counter(r.get("label_name") for r in valid)),
        "path": str(path),
    })

cc_summary_df = pd.DataFrame(cc_summary)
display(cc_summary_df)
write_json(cc_summary, CC_TOW_DIR / "cc_tow_summary.json")

for split in GENERATE_SPLITS:
    valid = [r for r in read_jsonl(CC_TOW_DIR / f"scicite_{split}_tow.jsonl") if validate_cc_tow_row(r)]
    if valid:
        print("\nExample", split)
        ex = valid[0]
        print("text:", ex["text"][:600])
        print("cue_words:", ex["cue_words"])
        print("tow:", ex["tow"])


## 9. Downstream notebooks

Use the generated CC-ToW files in the direct-input SciCite extension notebook. The clean IW-ToW dataset is generated separately by `03_scicite_inline_wordlevel_tow_generator.ipynb`, and the final evaluation notebooks consume these cached JSONL files.


In [ ]:
print("CC-ToW directory:", CC_TOW_DIR)
print("Raw SciCite data directory:", DATA_DIR)
